# BIOMASS tomogram — 3D visualisation

This notebook renders the same prepared TOM `DataCube` in three complementary ways: relative-power isosurfaces, normalized-contribution points, and the slider-driven interactive viewer. Install the optional dependencies with `pip install -e ".[visualisation]"`. Isosurfaces are scattering envelopes—not physical canopy boundaries.

In [1]:
from pathlib import Path
import sys

project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tomo_canopy').is_dir() and (p / 'data').is_dir())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tomo_canopy.cubegen import cube_from_layers
from tomo_canopy.processors import TomogramProcessor
from tomo_canopy.providers import TomogramProvider
from tomo_canopy.transforms import match_vertical_resolution
from tomo_canopy.visualisation import (
    CanopyViewer,
    InteractiveTomogramConfig,
    Tomogram3DConfig,
    TomogramIsosurfaceConfig,
)

## Open and prepare a bounded volume

The interactive viewer materializes its selected power array once, so always open a spatial and vertical subset. Resolution matching and power conversion are explicit preparation steps rather than hidden viewer behaviour.

In [2]:
plot_lon, plot_lat = 11.5730, -0.1750
delta = 0.006
bounds = (plot_lon - delta, plot_lat - delta, plot_lon + delta, plot_lat + delta)
tom_source = next((project_root / 'data' / 'TOM').glob('*L2A*'))

tom = TomogramProvider().open(
    tom_source, polarization='HV', height_range=(-20, 80), bounds=bounds
)
processor = TomogramProcessor()
powered = processor.to_power(tom)
prepared = match_vertical_resolution(
    powered, target_resolution=20.0, variables=['tomographic_power']
)
peak_power = float(prepared.tomographic_power.max().compute())
prepared = processor.to_db(
    prepared, variable='tomographic_power',
    reference=peak_power, floor_db=-12.0,
)
prepared = processor.integrate_vertical_bands(prepared, band_size=20.0)
prepared = processor.add_vertical_metrics(prepared, upper_height=40.0)
layer = processor.to_data_layer(prepared, name='lope_tomogram')
cube = cube_from_layers([layer])
viewer = CanopyViewer(cube)
prepared[['tomographic_power_db', 'vertical_contribution']]

<xarray.Dataset> Size: 384kB
Dimensions:                (polarization: 1, z: 51, y: 29, x: 29, height_band: 6)
Coordinates:
  * polarization           (polarization) <U2 8B 'HV'
  * z                      (z) float64 408B -20.0 -18.0 -16.0 ... 76.0 78.0 80.0
  * y                      (y) float64 232B -0.1809 -0.1804 ... -0.1696 -0.1692
  * x                      (x) float64 232B 11.57 11.57 11.57 ... 11.58 11.58
  * height_band            (height_band) float64 48B -20.0 0.0 ... 60.0 80.0
Data variables:
    tomographic_power_db   (polarization, z, y, x) float64 343kB dask.array<chunksize=(1, 48, 3, 3), meta=np.ndarray>
    vertical_contribution  (polarization, height_band, y, x) float64 40kB dask.array<chunksize=(1, 1, 3, 3), meta=np.ndarray>
Attributes: (12/25)
    coordinate_normalization:      XML latitude/longitude field semantics; ST...
    source:                        /home/simon/Documents/science/GFZ/projects...
    product_root:                  /home/simon/Documents/science/GFZ/projects...
    product_id:                    BIO_TP_TOM_L2A_20251007T050300_20251022T05...
    product_level:                 L2A
    modality:                      tomogram
    ...                            ...
    vertical_resolution_units:     m
    source_vertical_resolution:    0.0
    resolution_kernel:             gaussian_fwhm
    decibel_floor:                 -12.0
    vertical_structure_band_size:  20.0
    upper_contribution_height:     40.0

## Relative-power isosurfaces

The faint −8 dB envelope gives context, while −4 and −2 dB surfaces reveal progressively stronger connected scattering. Weak background is omitted instead of rendered as fog.

In [ ]:
plotter = viewer.tomogram_isosurfaces_3d(
    layer='lope_tomogram',
    variable='tomographic_power_db',
    polarization='HV',
    config=TomogramIsosurfaceConfig(
        levels=(-8.0, -4.0, -2.0),
        opacities=(0.10, 0.35, 0.85),
        vertical_exaggeration=12.0,
    ),
)
plotter.add_text('Nested HV scattering envelopes', font_size=11)
plotter.show()

Widget(value='<iframe src="http://localhost:35913/index.html?ui=P_0x7ceae1b4d8b0_0&reconnect=auto" class="pyvi…

JS Error => no webgl context
JS Error => TypeError: Cannot create proxy with a non-object as target or handler
JS Error => no webgl context
JS Error => TypeError: Cannot create proxy with a non-object as target or handler
JS Error => no webgl context
JS Error => TypeError: Cannot create proxy with a non-object as target or handler
JS Error => no webgl context
JS Error => TypeError: Cannot create proxy with a non-object as target or handler


## Normalized-contribution isosurfaces

These surfaces describe the fraction of each column's power assigned to a 20 m height band. They remove total-strength differences, so interpret them together with the relative-power view. Their stepped vertical form honestly reflects the coarse bands.

In [4]:
plotter = viewer.tomogram_isosurfaces_3d(
    layer='lope_tomogram',
    variable='vertical_contribution',
    polarization='HV',
    config=TomogramIsosurfaceConfig(
        levels=(0.10, 0.20, 0.35),
        opacities=(0.10, 0.35, 0.85),
        vertical_exaggeration=12.0,
    ),
)
plotter.add_text('HV contribution envelopes in 20 m bands', font_size=11)
plotter.show()

Widget(value='<iframe src="http://localhost:35913/index.html?ui=P_0x7ceae4b46690_1&reconnect=auto" class="pyvi…

## Height-coloured normalized contribution

Colour identifies height-band position and opacity follows contribution. This retains secondary vertical regions that a single isosurface threshold can hide.

In [5]:
plotter = viewer.tomogram_contribution_3d(
    layer='lope_tomogram',
    polarization='HV',
    config=Tomogram3DConfig(
        vertical_exaggeration=12.0, contribution_floor=0.02
    ),
)
plotter.add_text('20 m-band vertical contribution', font_size=11)
plotter.show()

Widget(value='<iframe src="http://localhost:35913/index.html?ui=P_0x7cea76e532f0_2&reconnect=auto" class="pyvi…

## Interactive scattering explorer

The controls change display filtering, normalization, height range, opacity, point size, and vertical exaggeration. Per-height-bin normalization emphasizes within-height spatial structure but removes absolute differences between heights; toggle it off when those differences matter.

In [6]:
interactive = viewer.interactive_tomogram(
    layer='lope_tomogram',
    polarization='HV',
    variable='tomographic_power',
    config=InteractiveTomogramConfig(
        column_quantile=0.80,
        normalization_quantile=0.95,
        vertical_exaggeration=12.0,
        point_opacity=0.35,
        point_size=15.0,
    ),
)
interactive.show(notebook=True)

Widget(value='<iframe src="http://localhost:35913/index.html?ui=P_0x7cea37c898e0_3&reconnect=auto" class="pyvi…